# Phishing Website Detection — Machine Learning Analysis & Benchmark

### College Capstone Project
**Objective:** Determine whether a given website URL is **Legitimate** or **Phishing** using passive lexical and structural feature extraction combined with supervised machine learning classification algorithms.

---

## 1. Import Libraries & Dependencies

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)

# Ensure project root is in sys.path
sys.path.append("..")
from utils.feature_extraction import extract_features_dict, get_feature_names

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully.")

## 2. Load & Explore the Benchmark Dataset

In [ ]:
data_path = "../data/dataset.csv"
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df["label_name"].value_counts())
df.head(10)

## 3. Extract Lexical & Structural URL Features

In [ ]:
feature_names = get_feature_names()
feature_rows = [extract_features_dict(str(u)) for u in df["url"]]

X = pd.DataFrame(feature_rows, columns=feature_names)
y = df["label"]

print(f"Extracted {X.shape[1]} features across {X.shape[0]} samples.")
X.describe()

## 4. Train/Test Stratified Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

## 5. Model Training & Algorithm Comparison
We compare 4 standard classification models:
1. **Logistic Regression** (Linear baseline)
2. **Decision Tree** (Non-linear single tree)
3. **Random Forest** (Ensemble bagging of 120 trees)
4. **Support Vector Machine** (Linear SVM with probability calibration)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=16, random_state=42, n_jobs=-1),
    "Support Vector Machine": CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2500, dual=False), cv=3)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    results.append({
        "Algorithm": name,
        "Accuracy": acc * 100,
        "Precision": prec * 100,
        "Recall": rec * 100,
        "F1-Score": f1 * 100
    })

metrics_df = pd.DataFrame(results)
metrics_df.sort_values(by="F1-Score", ascending=False)

## 6. Feature Importance (Random Forest)
Let's identify the most informative features used by the Random Forest classifier.

In [ ]:
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#0284c7')
plt.title("Random Forest - Relative Feature Importance for Phishing Detection", fontsize=13, fontweight='bold')
plt.xlabel("Gini Importance Score")
plt.tight_layout()
plt.show()

## 7. Confusion Matrix & ROC-AUC Analysis

In [ ]:
best_preds = rf_model.predict(X_test)
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Legitimate", "Phishing"], yticklabels=["Legitimate", "Phishing"])
plt.title("Confusion Matrix (Random Forest)", fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(y_test, best_preds, target_names=["Legitimate", "Phishing"]))

## 8. Conclusion & Viva Summary
- **Random Forest** achieved superior classification performance due to its ensemble bagging architecture, mitigating overfitting while capturing non-linear interactions across URL characteristics.
- Key predictive indicators include **IP address hosts**, **hyphen count**, **suspicious keywords**, and **subdomain nesting**.
- The trained model is exported via Joblib to `model/phishing_model.pkl` for low-latency web inference via the Flask REST API.